In [ ]:
from datetime import datetime
from enum import Enum
from pydantic import EmailStr
from sqlmodel import Column, Field, Relationship, SQLModel
from uuid import uuid4, UUID
from sqlalchemy.dialects import postgresql


class ShipmentStatus(str, Enum):
    placed = "placed"
    in_transit = "in_transit"
    out_for_delivery = "out_for_delivery"
    delivered = "delivered"


# Inherit SQLModel and set table = True
# to make a table in database
class Shipment(SQLModel, table = True):
    # Optional table name
    __tablename__ = "shipment"

    # Primary key with default value will be
    # assigned and incremented automatically
    id: UUID = Field(
        sa_column=Column(
            postgresql.UUID(as_uuid=True),
            primary_key=True,
            default=uuid4
        )
    )
    
    content: str
    weight: float = Field(le=25)
    destination: int
    status: ShipmentStatus
    estimated_delivery: datetime 
    seller_id: UUID = Field(foreign_key="seller.id")
    seller: "Seller" = Relationship(
        back_populates="shipments",
        sa_relationship_kwargs={"lazy": "selectin"}
    )

class Seller(SQLModel, table = True):
    
    id: UUID = Field(
        sa_column=Column(
            postgresql.UUID(as_uuid=True),
            primary_key=True,
            default=uuid4
        )
    )
    name: str

    email: EmailStr
    password_hash: str
    shipments: list[Shipment] = Relationship(
        back_populates="seller",
        sa_relationship_kwargs={"lazy": "selectin"}
    )

### UUID
UUID (Universally Unique Identifier) is a 128-bit unique identifier used to uniquely identify records in a database. Unlike traditional integer IDs that increase sequentially (1, 2, 3…), a UUID is generated as a random unique value, which makes it extremely unlikely for two records to have the same ID.

imports the UUID type and the uuid4() function. The function uuid4() generates a random unique ID every time a new record is created.

### sa_column=Column

sa_column=Column means that we are directly defining a SQLAlchemy column inside SQLModel.

Normally, SQLModel creates database columns automatically using Field(). However, when we need advanced control over the column (such as specifying a database-specific type, constraints, or default behavior), we can use sa_column=Column(...). This allows us to use the SQLAlchemy Column object directly.

### seller_id: int = Field(foreign_key="seller.id")
seller_id: int = Field(foreign_key="seller.id") defines a foreign key in the Shipment table. This means the seller_id column in the Shipment table references the id column in the Seller table. It ensures that every shipment is linked to a valid seller in the database.


### seller: "Seller" = Relationship(...)
seller: "Seller" = Relationship(...) creates an ORM relationship between the Shipment and Seller models. This allows us to access the related seller directly from a shipment object in Python, for example shipment.seller. The parameter back_populates="shipments" connects both models in two directions, so a seller can access all of its shipments using seller.shipments, while a shipment can access its seller using shipment.seller.

### sa_relationship_kwargs={"lazy": "selectin"}
The option sa_relationship_kwargs={"lazy": "selectin"} defines how related data is loaded from the database. The selectin strategy loads related records using an efficient secondary query, which improves performance when retrieving related objects.